In [1]:
import xarray as xr
import pandas as pd
import glob
import os
import math

import numpy as np
import re

In [2]:
ppe = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/simv4_iteration1.nc")
obs = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/obs.nc")

In [3]:

obs_dict = {"SWCF": "swcrf_toa", "LWCF": "lwcrf_toa", "TGCLDLWP": "clwp", "TMQ": "pwv",
          "CLDTOT_ISCCP": "clt_isccp", "FLUT": "olr", "PRECT": "pr","FSNTOA": "swabs_toa"}


In [4]:

lat_bins = np.arange(-85, 86, 5)  # -90 to 90 every 10 degrees
lat_labels = ((lat_bins[:-1] + lat_bins[1:]) / 2).astype(int).astype(str)  # midpoint labels
lat_labels = np.char.add("zonal_lat_",lat_labels)


In [5]:
len(lat_labels)

34

In [6]:
ppe_zonal_list = []
obs_zonal_list = []

for cam_name, obs_name in obs_dict.items():

    ppe_da = ppe[cam_name]
    filter_tf = obs[obs_name].notnull()
    
    ppe_da_filtered = ppe_da.where(filter_tf)
    
    zonal_ppe_temp = ppe_da_filtered.mean(dim  = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_dataframe().unstack(level = 1)
    zonal_ppe_temp.columns.name = None
    zonal_ppe_temp.columns = ["_".join(col) for col in list(zonal_ppe_temp.columns)]

    zonal_obs_temp = obs[obs_name].mean(dim = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_series()
    zonal_obs_temp.index = zonal_ppe_temp.columns

    
    ppe_zonal_list.append(zonal_ppe_temp)
    obs_zonal_list.append(zonal_obs_temp)

ppe_zonal_pd = pd.concat(ppe_zonal_list, axis = 1)
obs_zonal_pd = pd.concat(obs_zonal_list)


In [7]:
### Added locations for SWCF, LWCF, and PRECT
man_sel_locations1 = pd.Series({"cli": "SWCF", "lat_min": -6,"lat_max": -4, "lon_min":  141, "lon_max": 144})
man_sel_locations2 = pd.Series({"cli": "SWCF", "lat_min": 7,"lat_max": 9, "lon_min":  235, "lon_max": 240})



manul_ppe_info = pd.concat([man_sel_locations1, man_sel_locations2], axis  = 1).transpose()
manul_ppe_info

,cli,lat_min,lat_max,lon_min,lon_max
0,SWCF,-6,-4,141,144
1,SWCF,7,9,235,240


In [8]:
ppe_manual_list = []
obs_manual_list = []
manual_name_list = []

for row_ind, row in manul_ppe_info.iterrows():
    
    temp_obs = obs[obs_dict[row.cli]].sel(lat = slice(row.lat_min, row.lat_max), lon = slice(row.lon_min, row.lon_max)).mean(dim = ["lat", "lon"]).values
    if ~np.isnan(temp_obs):
        temp_ppe = ppe[row.cli].sel(lat = slice(row.lat_min, row.lat_max), lon = slice(row.lon_min, row.lon_max)).mean(dim = ["lat", "lon"]).to_dataframe()
        
        manual_name_list.append("_".join(row.astype(str)))
        ppe_manual_list.append(temp_ppe)
        obs_manual_list.append(temp_obs)
    else:
        print("??")


In [9]:
ppe_manual_list = pd.concat(ppe_manual_list,axis = 1)
obs_manual_list = pd.Series(obs_manual_list)

ppe_manual_list.columns = manual_name_list
obs_manual_list.index = manual_name_list

In [10]:
ppe_pd = pd.concat([ppe_zonal_pd, ppe_manual_list], axis = 1)
obs_pd = pd.concat([obs_zonal_pd, obs_manual_list])


In [12]:
#ppe_pd.to_csv("/glade/work/qingyuany/repo_data/spatialtuning/ppe_zonal_v4_iteration1.csv", index = True)
#obs_pd.to_csv("/glade/work/qingyuany/repo_data/spatialtuning/obs_zonal_v4_iteration1.csv", index = True)
